# 0. Install and import the required dependencies

**You may add or remove based on your assigned model!**

In [ ]:
!pip install -U tokenizers transformers accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata

# 1. Authentication & Model selection

**Retrieve the Hugging Face token securely from Colab's "Secrets" tab (the key icon on the left).**

In [ ]:
try:
    hf_token = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("WARNING: 'HF_TOKEN' not found in Colab Secrets.")
    hf_token = None

**CHANGE THIS TO YOUR ASSIGNED MODEL**

In [ ]:
# A smaller model that fits on a free Colab GPU
model_name = "ibm-granite/granite-3.3-8b-instruct"

# 2. Hardware optimization (Quantization)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # What is loaded in 4 bit? why?
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 3. Load the tokenizer & model

In [ ]:
print(f"Loading Tokenizer for {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True # Does your model need it?
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # What about right?

In [ ]:
print("Loading Model on Colab T4 GPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    trust_remote_code=True, # Does your model need it?
    quantization_config=bnb_config, # from section 2 above
    torch_dtype=torch.float16
)

# 4. INFERENCE & HYPERPARAMETER TUNING

**Design the prompt (Does this design/technique have a name?)**

In [ ]:
# ============================================================
# Context-Aware Prompting for Architectural Recovery
# Hadoop MapReduce Cluster Analysis using LLM  for IL 30
# ============================================================

# 1. Define the Context-Aware Prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are a software architecture recovery expert specializing in "
            "large-scale distributed systems such as Hadoop MapReduce. "
            "You are given classes that were automatically grouped into the "
            "same architectural cluster using an RSF-based clustering algorithm. "
            "The classes in the cluster are likely to collaborate to provide "
            "a shared subsystem or architectural responsibility. "
            "Your task is to infer the architectural intent of the cluster "
            "by analyzing naming patterns, package structures, responsibilities, "
            "and interactions between the classes."
        )
    },

    {
        "role": "user",
        "content": """
The following classes were recovered as part of the SAME architectural cluster
during Hadoop MapReduce architectural recovery.

### Cluster Context:
These classes appear to coordinate job submission, job execution,
cluster communication, security token handling, distributed cache
management, and MapReduce client-side interaction.

### Classes in the Cluster:
- org.apache.hadoop.mapreduce.Cluster
- org.apache.hadoop.mapreduce.CryptoUtils
- org.apache.hadoop.mapreduce.Job
- org.apache.hadoop.mapreduce.Job$12
- org.apache.hadoop.mapreduce.JobContext
- org.apache.hadoop.mapreduce.JobResourceUploader
- org.apache.hadoop.mapreduce.JobStatus
- org.apache.hadoop.mapreduce.JobSubmitter
- org.apache.hadoop.mapreduce.JobSubmitter$1
- org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
- org.apache.hadoop.mapreduce.TaskCompletionEvent
- org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
- org.apache.hadoop.mapreduce.protocol.ClientProtocol
- org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
- org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
- org.apache.hadoop.mapreduce.task.JobContextImpl
- org.apache.hadoop.mapreduce.tools.CLI

### Architectural Recovery Task:
Using the cluster context above, infer the architectural purpose
of this subsystem.

Provide:

1. Architectural Title
   - A concise subsystem/component name.

2. Architectural Summary
   - 4-6 sentences describing:
     - the main responsibility of the cluster,
     - how the classes collaborate,
     - what MapReduce functionality they support,
     - how communication/security/resource management are handled.

3. Key Architectural Responsibilities
   - Bullet list of major subsystem responsibilities.

4. Inter-Class Collaboration Insights
   - Explain how important classes interact with each other.

5. Dominant Architectural Concern
   - State whether the cluster mainly represents:
     - Job Management
     - Communication
     - Security
     - Resource Coordination
     - Client API Infrastructure
     - Execution Monitoring
     - or a combination of these.

Focus on architectural interpretation rather than line-by-line code explanation.
"""
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

In [ ]:
# ============================================================
# Context-Aware Prompting using RSF Architectural Clusters for IL 30
# ============================================================

messages = [

    # ========================================================
    # SYSTEM PROMPT
    # ========================================================
    {
        "role": "system",
        "content": (
            "You are an expert in software architecture recovery and "
            "distributed systems analysis. "
            "You are analyzing Hadoop MapReduce architectural clusters "
            "generated using an RSF-based clustering algorithm. "
            "Classes grouped in the same cluster are assumed to collaborate "
            "to implement a common subsystem responsibility. "
            "Your task is to infer the architectural intent, subsystem role, "
            "and collaboration patterns of the cluster."
        )
    },

    # ========================================================
    # USER PROMPT
    # ========================================================
    {
        "role": "user",
        "content": """
The following classes were grouped together in the SAME architectural cluster
(cluster id = 1) during Hadoop MapReduce architectural recovery.

### Architectural Cluster Context
This cluster appears to contain classes related to:
- Job submission
- Client-side MapReduce APIs
- Task/job context management
- Distributed cache management
- Job split handling
- Input/output coordination
- Security token handling
- Job control and execution coordination

### Classes in Cluster 1

org.apache.hadoop.mapreduce.Cluster
org.apache.hadoop.mapreduce.CryptoUtils
org.apache.hadoop.mapreduce.Job
org.apache.hadoop.mapreduce.JobACL
org.apache.hadoop.mapreduce.JobContext
org.apache.hadoop.mapreduce.JobID
org.apache.hadoop.mapreduce.JobResourceUploader
org.apache.hadoop.mapreduce.JobStatus
org.apache.hadoop.mapreduce.JobSubmitter
org.apache.hadoop.mapreduce.MapContext
org.apache.hadoop.mapreduce.ReduceContext
org.apache.hadoop.mapreduce.TaskAttemptContext
org.apache.hadoop.mapreduce.TaskAttemptID
org.apache.hadoop.mapreduce.TaskID
org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
org.apache.hadoop.mapreduce.filecache.DistributedCache
org.apache.hadoop.mapreduce.protocol.ClientProtocol
org.apache.hadoop.mapreduce.security.TokenCache
org.apache.hadoop.mapreduce.split.JobSplit
org.apache.hadoop.mapreduce.split.JobSplitWriter
org.apache.hadoop.mapreduce.task.JobContextImpl
org.apache.hadoop.mapreduce.task.TaskAttemptContextImpl
org.apache.hadoop.mapreduce.tools.CLI
org.apache.hadoop.mapreduce.lib.input.FileInputFormat
org.apache.hadoop.mapreduce.lib.output.FileOutputFormat
org.apache.hadoop.mapreduce.lib.jobcontrol.JobControl
org.apache.hadoop.mapreduce.lib.jobcontrol.ControlledJob

### Additional Context
Nearby clusters contain:
- Cluster 2 → Counters, JobHistory, Shuffle, Metrics, Security Utilities
- Cluster 3 → Mapper/Reducer chaining and execution pipeline
- Cluster 4 → Counter framework implementation
- Cluster 5 → Counter factory and initialization logic

This indicates that Cluster 1 is likely responsible for the main
client-side job coordination and execution infrastructure.

### Task

Analyze this cluster and provide:

1. Architectural Title
2. High-Level Architectural Summary (4-6 sentences)
3. Main Architectural Responsibilities
4. Important Class Collaboration Patterns
5. Dominant Architectural Concern
6. Why these classes likely belong together architecturally

Focus on subsystem-level interpretation rather than source-code details.
"""
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

In [ ]:
# ============================================================
# Few-Shot Prompting (Example-Based)
# Architectural Recovery using Hadoop RSF Clusters  for IL 50
# ============================================================

messages = [

    # ========================================================
    # SYSTEM ROLE
    # ========================================================
    {
        "role": "system",
        "content": (
            "You are an expert in software architecture recovery and "
            "distributed systems analysis. "
            "You analyze RSF-based architectural clusters from Hadoop "
            "MapReduce and infer subsystem responsibilities, collaboration "
            "patterns, and architectural intent."
        )
    },

    # ========================================================
    # FEW-SHOT EXAMPLE 1
    # ========================================================
    {
        "role": "user",
        "content": """
### Example Cluster

Cluster ID: 10

Classes:
- org.apache.hadoop.fs.FileSystem
- org.apache.hadoop.fs.Path
- org.apache.hadoop.fs.FileStatus
- org.apache.hadoop.fs.BlockLocation
- org.apache.hadoop.fs.FSDataInputStream

Analyze this architectural cluster.
"""
    },

    {
        "role": "assistant",
        "content": """
Architectural Title:
Distributed File System Access Layer

Architectural Summary:
This cluster represents Hadoop's distributed file system abstraction subsystem.
The classes collaborate to provide file access, metadata management, block-level
storage handling, and distributed input stream operations. FileSystem acts as
the primary interface for interacting with HDFS, while Path and FileStatus
manage file identification and metadata retrieval. BlockLocation supports
distributed block awareness, and FSDataInputStream enables scalable reading
from distributed storage nodes.

Main Responsibilities:
- Distributed file access
- File metadata handling
- Block-level storage abstraction
- Stream-based data reading

Dominant Architectural Concern:
Storage and File System Management
"""
    },

    # ========================================================
    # FEW-SHOT EXAMPLE 2
    # ========================================================
    {
        "role": "user",
        "content": """
### Example Cluster

Cluster ID: 12

Classes:
- org.apache.hadoop.mapreduce.Mapper
- org.apache.hadoop.mapreduce.Reducer
- org.apache.hadoop.mapreduce.Partitioner
- org.apache.hadoop.mapreduce.RecordReader
- org.apache.hadoop.mapreduce.RecordWriter
- org.apache.hadoop.mapreduce.InputSplit

Analyze this architectural cluster.
"""
    },

    {
        "role": "assistant",
        "content": """
Architectural Title:
MapReduce Execution Pipeline

Architectural Summary:
This cluster represents the core MapReduce data processing pipeline.
The classes collaborate to execute distributed computation tasks using
the MapReduce programming model. Mapper and Reducer implement transformation
and aggregation logic, while Partitioner controls key distribution across reducers.
RecordReader and RecordWriter handle data ingestion and output generation,
and InputSplit enables parallel processing of distributed datasets.

Main Responsibilities:
- Distributed data processing
- Parallel task execution
- Input/output data handling
- Key partitioning and aggregation

Dominant Architectural Concern:
Execution and Data Processing
"""
    },

    # ========================================================
    # TARGET CLUSTER
    # ========================================================
    {
        "role": "user",
        "content": """
### Target Cluster

Cluster ID: 1

Classes:
- org.apache.hadoop.mapreduce.Cluster
- org.apache.hadoop.mapreduce.CryptoUtils
- org.apache.hadoop.mapreduce.Job
- org.apache.hadoop.mapreduce.JobACL
- org.apache.hadoop.mapreduce.JobContext
- org.apache.hadoop.mapreduce.JobID
- org.apache.hadoop.mapreduce.JobResourceUploader
- org.apache.hadoop.mapreduce.JobStatus
- org.apache.hadoop.mapreduce.JobSubmitter
- org.apache.hadoop.mapreduce.MapContext
- org.apache.hadoop.mapreduce.ReduceContext
- org.apache.hadoop.mapreduce.TaskAttemptContext
- org.apache.hadoop.mapreduce.TaskAttemptID
- org.apache.hadoop.mapreduce.TaskID
- org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
- org.apache.hadoop.mapreduce.filecache.DistributedCache
- org.apache.hadoop.mapreduce.protocol.ClientProtocol
- org.apache.hadoop.mapreduce.security.TokenCache
- org.apache.hadoop.mapreduce.split.JobSplit
- org.apache.hadoop.mapreduce.split.JobSplitWriter
- org.apache.hadoop.mapreduce.task.JobContextImpl
- org.apache.hadoop.mapreduce.task.TaskAttemptContextImpl
- org.apache.hadoop.mapreduce.tools.CLI
- org.apache.hadoop.mapreduce.lib.input.FileInputFormat
- org.apache.hadoop.mapreduce.lib.output.FileOutputFormat
- org.apache.hadoop.mapreduce.lib.jobcontrol.JobControl
- org.apache.hadoop.mapreduce.lib.jobcontrol.ControlledJob

Analyze this architectural cluster using the SAME format as the examples above.
"""
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

In [ ]:
# ============================================================
# Few-Shot Prompting for Hadoop Architectural Recovery for IL 50
# ============================================================

messages = [

    {
        "role": "system",
        "content": (
            "You are an expert in software architecture recovery and "
            "distributed systems analysis. "
            "You analyze Hadoop MapReduce architectural clusters "
            "and infer subsystem responsibilities."
        )
    },

    # ========================================================
    # EXAMPLE 1
    # ========================================================

    {
        "role": "user",
        "content": """
### Example Cluster

Classes:
- org.apache.hadoop.fs.FileSystem
- org.apache.hadoop.fs.Path
- org.apache.hadoop.fs.FileStatus
- org.apache.hadoop.fs.BlockLocation

Analyze this cluster.
"""
    },

    {
        "role": "assistant",
        "content": """
Architectural Title:
Distributed File System Layer

Architectural Summary:
This cluster represents Hadoop's distributed storage abstraction subsystem.
The classes collaborate to provide file access, metadata handling,
and block-level storage management across distributed nodes.

Main Responsibilities:
- File access
- Metadata management
- Distributed storage handling
"""
    },

    # ========================================================
    # EXAMPLE 2
    # ========================================================

    {
        "role": "user",
        "content": """
### Example Cluster

Classes:
- org.apache.hadoop.mapreduce.Mapper
- org.apache.hadoop.mapreduce.Reducer
- org.apache.hadoop.mapreduce.InputSplit
- org.apache.hadoop.mapreduce.RecordReader

Analyze this cluster.
"""
    },

    {
        "role": "assistant",
        "content": """
Architectural Title:
MapReduce Processing Pipeline

Architectural Summary:
This cluster represents the core distributed data processing subsystem.
The classes collaborate to execute parallel map and reduce operations
while handling distributed input processing.

Main Responsibilities:
- Distributed computation
- Parallel task execution
- Input data processing
"""
    },

    # ========================================================
    # TARGET CLUSTER
    # ========================================================

    {
        "role": "user",
        "content": """
### Target Cluster

Classes:
- org.apache.hadoop.mapreduce.Cluster
- org.apache.hadoop.mapreduce.CryptoUtils
- org.apache.hadoop.mapreduce.Job
- org.apache.hadoop.mapreduce.Job$12
- org.apache.hadoop.mapreduce.JobContext
- org.apache.hadoop.mapreduce.JobResourceUploader
- org.apache.hadoop.mapreduce.JobStatus
- org.apache.hadoop.mapreduce.JobSubmitter
- org.apache.hadoop.mapreduce.JobSubmitter$1
- org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
- org.apache.hadoop.mapreduce.TaskCompletionEvent
- org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
- org.apache.hadoop.mapreduce.protocol.ClientProtocol
- org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
- org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
- org.apache.hadoop.mapreduce.task.JobContextImpl
- org.apache.hadoop.mapreduce.tools.CLI

Analyze this architectural cluster using the SAME format as the examples above.
"""
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())

In [ ]:
# ============================================================
# Context-Aware Prompting for Hadoop Architectural Recovery
# ============================================================

messages = [

    {
        "role": "system",
        "content": (
            "You are an expert in software architecture recovery and "
            "distributed systems analysis. "
            "You analyze Hadoop MapReduce architectural clusters generated "
            "using RSF clustering techniques. "
            "Classes grouped together are assumed to collaborate to implement "
            "a common subsystem responsibility."
        )
    },

    {
        "role": "user",
        "content": """
The following classes belong to the SAME architectural cluster
(cluster id = 1) identified during Hadoop MapReduce recovery.

### Cluster Context

This cluster appears related to:
- Job submission
- Cluster communication
- Distributed cache handling
- Job execution coordination
- Client-side MapReduce APIs
- Security token management

### Classes in Cluster 1

- org.apache.hadoop.mapreduce.Cluster
- org.apache.hadoop.mapreduce.CryptoUtils
- org.apache.hadoop.mapreduce.Job
- org.apache.hadoop.mapreduce.Job$12
- org.apache.hadoop.mapreduce.JobContext
- org.apache.hadoop.mapreduce.JobResourceUploader
- org.apache.hadoop.mapreduce.JobStatus
- org.apache.hadoop.mapreduce.JobSubmitter
- org.apache.hadoop.mapreduce.JobSubmitter$1
- org.apache.hadoop.mapreduce.JobSubmitter$SplitComparator
- org.apache.hadoop.mapreduce.TaskCompletionEvent
- org.apache.hadoop.mapreduce.filecache.ClientDistributedCacheManager
- org.apache.hadoop.mapreduce.protocol.ClientProtocol
- org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSecretManager
- org.apache.hadoop.mapreduce.security.token.delegation.DelegationTokenSelector
- org.apache.hadoop.mapreduce.task.JobContextImpl
- org.apache.hadoop.mapreduce.tools.CLI

### Additional Architectural Context

Nearby clusters contain:
- Counters framework
- Task execution contexts
- MapReduce pipeline components
- Output formatting infrastructure

This suggests Cluster 1 represents the central client-side
job coordination and submission subsystem.

### Task

Analyze this cluster and provide:

1. Architectural Title
2. High-Level Architectural Summary
3. Main Responsibilities
4. Collaboration Between Important Classes
5. Dominant Architectural Concern

Focus on subsystem-level architectural interpretation.
"""
    }
]

# 2. Tokenize the messages
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

# 3. Generate the response with optimized parameters
print("Generating the optimized architectural analysis...\n")
outputs = model.generate(
    **inputs,
    max_new_tokens=350,   # Ensures the model doesn't cut off mid-sentence
    do_sample=False,      # Uses Greedy Search for technical precision and consistency
    pad_token_id=tokenizer.eos_token_id
)

# 4. Display the result
decoded_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(decoded_output.split("assistant")[-1].strip())